# Linear Probing Shuffled-Ident Baseline

This notebook builds a random baseline by shuffling the `ident -> target` mapping,
then runs linear probes on that shuffled target assignment.


In [1]:
import os
from pathlib import Path
import pandas as pd

from prob.paths_and_io import get_project_root, load_X_from_pt
from prob.prob_config import get_ds_load_config
from prob.prob_models_and_run import linear_models_shuffled_ident_baseline


In [12]:
project_root = get_project_root()
os.environ["HOME_PROJ_DIR"] = str(project_root)
print("Project root:", project_root)

# ---- Edit these only ----
TARGET_FILE = "weighted_hb_score.pt"
LAYER_NUM = 3
RANDOM_STATE = 96
BASELINE_TAG = "shuffled_ident"

# Cluster-safe CPU budget
CPU_BUDGET = int(
    os.getenv("SLURM_CPUS_PER_TASK")
    or os.getenv("NSLOTS")
    or os.getenv("OMP_NUM_THREADS")
    or 1
)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
N_JOBS = CPU_BUDGET
print(f"CPU_BUDGET={CPU_BUDGET}, N_JOBS={N_JOBS}")

# Optional: re-use best params for faster reruns.
REUSE_BEST_PARAMS = True
BEST_PARAMS_CACHE_DIR = project_root / "data" / "probing" / "_shared_best_params"


Project root: /home/famo00001/kinodata-3D-affinity-prediction
CPU_BUDGET=1, N_JOBS=1


In [13]:
prob_config = get_ds_load_config(target_file=TARGET_FILE)
X = load_X_from_pt(prob_config.output_dir, layer_num=LAYER_NUM)
prob_config.update({"layer_num": LAYER_NUM}, allow_duplicates=True)

print("X shape:", X.shape)
print("target file:", TARGET_FILE)
print("layer:", LAYER_NUM)


X shape: (41238, 256)
target file: weighted_hb_score.pt
layer: 3


In [14]:
runs = linear_models_shuffled_ident_baseline(
    prob_config,
    X,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
    baseline_tag=BASELINE_TAG,
    target_file=TARGET_FILE,
    reuse_best_params=REUSE_BEST_PARAMS,
    best_params_cache_dir=BEST_PARAMS_CACHE_DIR,
)

baseline_df = pd.DataFrame(runs).sort_values(["r2", "rmse"], ascending=[False, True]).reset_index(drop=True)
baseline_df


,experiment,layer,r2,rmse,mae,fit_seconds,model__alpha
0,weighted_hb_score_shuffled_ident_lasso,3,-0.006198,4.252255,3.402646,70.696384,0.00001
1,weighted_hb_score_shuffled_ident_ridge,3,-0.006211,4.252283,3.402661,0.229554,0.00100


In [15]:
display(baseline_df[["experiment", "layer", "r2", "rmse", "mae", "fit_seconds"]])
best = baseline_df.iloc[0]
print(f"Best shuffled-ident baseline: {best['experiment']} | R2={best['r2']:.4f} | RMSE={best['rmse']:.4f}")


,experiment,layer,r2,rmse,mae,fit_seconds
0,weighted_hb_score_shuffled_ident_lasso,3,-0.006198,4.252255,3.402646,70.696384
1,weighted_hb_score_shuffled_ident_ridge,3,-0.006211,4.252283,3.402661,0.229554


Best shuffled-ident baseline: weighted_hb_score_shuffled_ident_lasso | R2=-0.0062 | RMSE=4.2523


In [16]:
baseline_target_name = f"{Path(TARGET_FILE).stem}_{BASELINE_TAG}"
out_csv = Path(prob_config.output_dir) / baseline_target_name / "experiments" / f"layer_{LAYER_NUM}_linear_baseline_summary.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
baseline_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)


Saved: /home/famo00001/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/rmsd_cutoff_2/random-k-fold/weighted_hb_score_shuffled_ident/experiments/layer_3_linear_baseline_summary.csv
